In [1]:
fiber_files = [f'sox2_fibers/fiber_sox2_{i}_{j}.h5' for j in range(1,4) for i in (19,39,50)]
traj_files = [f'MC_1_trajectories/sox2_traj_{i}_{j}.h5' for j in range(1,4) for i in (19,39,50)]

In [14]:
with open('run.sh','r') as f:
    bp = f.read()
for t_file,f_file in zip(traj_files,fiber_files):
    with open(f'MC_1_run_files/run_{f_file[-12:-3]}.sh','w') as f:
        f.write(bp[:-8]+ f'{t_file} --h5file {f_file}')

In [21]:
for name in fiber_files:
    !sbatch MC_1_run_files/run_{name[-12:-3]}.sh

Submitted batch job 21965
Submitted batch job 21966
Submitted batch job 21967
Submitted batch job 21968
Submitted batch job 21969
Submitted batch job 21970
Submitted batch job 21971
Submitted batch job 21972
Submitted batch job 21973


In [2]:
!sbatch MC_1_run_files/run_{fiber_files[0][-12:-3]}.sh

Submitted batch job 21974


In [6]:
!squeue

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
             21833    cryoem cryospar cryospar  R   18:25:42      1 node01
             21163       gpu       wt afedulov  R 41-17:03:17      1 node02
             21704       gpu linker30 afedulov  R 6-12:55:15      1 node08
             21705       gpu linker30 afedulov  R 6-12:53:25      1 node08
             21709       gpu prechrom afedulov  R 6-08:30:28      1 node03
             21710       gpu prechrom afedulov  R 6-08:30:15      1 node03
             21821       gpu KCNA3-PO vnovosel  R   21:28:35      1 node02
             21966       gpu CG_fiber vvasilev  R       2:47      1 node09
             21967       gpu CG_fiber vvasilev  R       2:47      1 node10
             21968       gpu CG_fiber vvasilev  R       2:46      1 node10
             21969       gpu CG_fiber vvasilev  R       2:44      1 node05
             21970       gpu CG_fiber vvasilev  R       2:44      1 node05
             2

In [2]:
import h5py

In [3]:
%load_ext autoreload
%autoreload 2
from sys import path

path.insert(0,'../PyNAMod/')
import pynamod

/home/vvasilev/.conda/envs/pynamod/lib/python3.11/site-packages/nglview/__init__.py:12: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources
/home/vvasilev/.conda/envs/pynamod/lib/python3.11/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(
/home/vvasilev/SOX2_Runs/../PyNAMod/pynamod/structures/DNA_structure.py:260: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted as an index
  self.movable_steps = torch.tensor(data['movable_steps'])
/home/vvasilev/SOX2_Runs/../PyNAMod/pynamod

In [4]:
import warnings
warnings.filterwarnings('ignore')

In [5]:
cgs = pynamod.CG_Structure()
file = h5py.File(fiber_files[0],'r')
cgs.load_from_h5(file)
cgs.dna.transfer_trajectory_to_h5(traj_files[0],'w')

In [9]:
en = pynamod.Energy()
en.set_energy_matrices(cgs,ignore_neighbors=10)

intg = pynamod.Iterator(cgs,en,sigma_rot=0.15,sigma_transl=0.3)
intg.run(target_accepted_steps=10000,max_steps=100000,device='cuda',KT_factor=0.6,save_every=5,transfer_to_memory_every=50)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument mat2 in method wrapper_CUDA_mm)